# tanque-cheio · 05 — modelos maiores no dev2 (M14)

Compara, **sem ajuste e com o mesmo prompt único** (D26), o Qwen2.5-Coder-1.5B
(base e ft-v1) com os Qwen2.5-Coder-7B, 14B e 32B-Instruct no `datasets/dev2.jsonl`
(113 perguntas), numa sessão com **2 GPUs T4** do Kaggle:

| Sistema | Precisão | GPUs |
|---|---|---|
| `base-qwen2.5-coder-1.5b-instruct` | fp16 | 1 |
| `ft-qwen2.5-coder-1.5b-instruct` (adapter v1 do HF, fundido) | fp16 | 1 |
| `base-qwen2.5-coder-7b-instruct` | nf4 (4 bits, D27) | 1 |
| `base-qwen2.5-coder-14b-instruct` | nf4 | 1 |
| `base-qwen2.5-coder-32b-instruct` | nf4 | 2 (`device_map="auto"`), **por último** |

Regras da v2: decodificação gulosa, `max_new_tokens = 384` (D31), autocorreção com
1 nova tentativa (D32). Cada modelo grava, numa só execução,
`<sistema>__dev2.jsonl` (single-shot) e `<sistema>+retry__dev2.jsonl`, mais um
`run-info__...json` para cada um (commit do checkpoint, `quantization`, `gpus`,
`peak_gpu_mem_gib`, tempo). Os checkpoints ficam fixados por commit na 1ª célula.

**Defesas (regra 0.8 e D37):** sem restringir a 1 GPU (o 32B precisa das 2): os
modelos que cabem numa T4 usam `device_map={"": 0}`. O `torchao` é removido e
`is_torchao_available()` é checado antes de tudo. Entre um modelo e outro, a memória
da GPU e o cache do Hugging Face são liberados. O 32B tem limite de
`TIME_BUDGET_32B_H` horas e checkpoint por item: se estourar o tempo, faltar
memória, faltar disco ou a sessão tiver só 1 GPU, ele para (ou é pulado) e isso fica
no run-info. Se o disco não comportar os 61 GiB do 32B em bf16, o notebook usa o
checkpoint reserva já quantizado em nf4 (18 GiB) e registra `model_source = "fallback"`.

**No Kaggle:** Settings → Accelerator → **GPU T4 x2**; **Internet on**. *Save Version →
Save & Run All*. Pela CLI, `machine_shape: "NvidiaTeslaT4"` já é "GPU T4 ×2".
Tempo estimado: ~3 h (pessimista ~5 h); detalhes em `results/large-baselines.md`.
Para dividir em 2 execuções, `PART = "a"` (1,5B, ft, 7B, 14B) e depois `PART = "b"`
(32B).

As métricas não são calculadas aqui: baixe os JSONL e pontue no computador (passos em
`notebooks/README.md`, seção M14).

In [ ]:
# Parâmetros: edite só esta célula.
REPO_REF = "main"  # branch, tag ou commit do tanque-cheio
DATASET = "dev2"  # decisões da v2 só no dev (regra 0.6); o teste fica para o M17
LIMIT = None  # ex.: 3 para um ensaio rápido (os arquivos ganham o sufixo -sample3)
MAX_NEW_TOKENS = 384  # D31
RETRY = True  # D32: grava também <sistema>+retry__<split>.jsonl
PART = "all"  # "all"; ou "a" (1,5B, ft, 7B, 14B) e depois "b" (32B), em 2 versões
TIME_BUDGET_32B_H = 3  # D37: o 32B para depois disso (download e carga inclusos)
# Saída de uma versão anterior anexada como input (Add Input → Notebook Output), para
# continuar de onde parou: ex. "/kaggle/input/tanque-cheio-large". None = do zero.
RESUME_DIR = None
DTYPE = "float16"  # a T4 não tem bf16 nativo; também é o dtype de computação do nf4
OUT_DIR = "/kaggle/working"  # vira a saída do notebook no Kaggle
REPO_DIR = "/tmp/tanque-cheio"  # fora de OUT_DIR, para não ir para a saída

# Checkpoints fixados por commit (https://huggingface.co/api/models/<repo>, 29/10/2026).
# O 1,5B e o adapter são os mesmos do 04_test_predict (M7).
MODELS = [
    {
        "model_id": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
        "revision": "2e1fd397ee46e1388853d2af2c993145b0f1098a",
        "quantization": None,  # fp16, como na v1
        "gpus": 1,
        "part": "a",
    },
    {
        "model_id": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
        "revision": "2e1fd397ee46e1388853d2af2c993145b0f1098a",
        "adapter_id": "belasoares/tanque-cheio-qwen2.5-coder-1.5b-lora",  # ft-v1 (M6)
        "adapter_revision": "bd8079ff5467c339586bc7fc31f18a19b6509a9e",
        "quantization": None,  # fp16, adapter fundido
        "gpus": 1,
        "part": "a",
    },
    {
        "model_id": "Qwen/Qwen2.5-Coder-7B-Instruct",
        "revision": "c03e6d358207e414f1eca0bb1891e29f1db0e242",
        "quantization": "nf4",
        "gpus": 1,
        "part": "a",
    },
    {
        "model_id": "Qwen/Qwen2.5-Coder-14B-Instruct",
        "revision": "aedcc2d42b622764e023cf882b6652e646b95671",
        "quantization": "nf4",
        "gpus": 1,
        "part": "a",
    },
    {
        "model_id": "Qwen/Qwen2.5-Coder-32B-Instruct",
        "revision": "381fc969f78efac66bc87ff7ddeadb7e73c218a7",
        "quantization": "nf4",
        "gpus": 2,
        "max_memory": {0: "13GiB", 1: "14GiB"},  # folga na GPU 0 para ativações
        "part": "b",
        "time_budget_h": TIME_BUDGET_32B_H,
        # Só se o disco não comportar os 61 GiB em bf16: o mesmo modelo já em nf4 com
        # quantização dupla (18 GiB, Apache-2.0); a computação continua em fp16.
        "fallback": {
            "model_id": "unsloth/Qwen2.5-Coder-32B-Instruct-bnb-4bit",
            "revision": "89e52be6bf3252ffa41a6db933aeba59230848dd",
        },
    },
]

In [ ]:
# Instala o pacote `tanque` no REPO_REF (com as versões do M6/M7 e o bitsandbytes do
# QLoRA) e clona o repositório no mesmo ref, para ter datasets/, configs/ e snapshot/.
#
# ATENÇÃO: aqui NÃO se define CUDA_VISIBLE_DEVICES = "0" (ao contrário do 02 e do 04):
# o 32B precisa das 2 T4. Os modelos que cabem numa GPU usam device_map={"": 0}.
import os
import subprocess
import sys

REPO_URL = "https://github.com/belasoaress/tanque-cheio"
# As mesmas versões do fine-tuning e do teste (uv.lock); torch vem da imagem.
PINS = [
    "transformers==5.21.0",
    "peft==0.21.2",
    "accelerate==1.15.0",
    "bitsandbytes==0.50.2",  # 4 bits (nf4); só existe com GPU
]


def sh(*cmd: str) -> None:
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


sh(sys.executable, "-m", "pip", "install", "-q", f"tanque @ git+{REPO_URL}@{REPO_REF}", *PINS)
# A imagem do Kaggle traz um torchao antigo (0.10) e o peft se recusa a carregar o
# adapter quando ele está instalado. O projeto não usa torchao: remove.
sh(sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao")
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "origin", REPO_REF)
sh("git", "-C", REPO_DIR, "checkout", "--quiet", "FETCH_HEAD")
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True
).strip()
os.environ["TANQUE_ROOT"] = REPO_DIR  # antes de importar `tanque`
print("tanque-cheio em", REPO_COMMIT)

In [ ]:
import torch
import transformers
from peft.import_utils import is_torchao_available

print("torch", torch.__version__, "| transformers", transformers.__version__)
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(N_GPUS)]
print("GPUs visíveis:", N_GPUS, GPU_NAMES)
if N_GPUS < 2:
    print(
        "Aviso: menos de 2 GPUs. Escolha Settings > Accelerator > GPU T4 x2; "
        "sem isso o 32B é pulado (D37)."
    )

# Falha cedo (antes da inferência) se o peft não puder carregar adapters.
is_torchao_available()  # levanta ImportError se houver um torchao incompatível
if N_GPUS:
    import bitsandbytes

    print("bitsandbytes", bitsandbytes.__version__)

In [ ]:
# Banco DuckDB do snapshot versionado (D25), para a autocorreção executar as SQLs
# (~6 s, sem download). Equivale a `tanque data build --from-parquet ...`.
sh(
    sys.executable,
    "-c",
    "from tanque.cli import app; app()",
    "data",
    "build",
    "--from-parquet",
    os.path.join(REPO_DIR, "snapshot", "coletas-2026-09.parquet"),
)

In [ ]:
# Continuação de uma versão anterior (opcional): copia os JSONL parciais para OUT_DIR;
# run_predictions(resume=True) pula os itens que já estão lá.
import shutil
from pathlib import Path

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
if RESUME_DIR:
    for src in sorted(Path(RESUME_DIR).rglob("*.jsonl")):
        dst = Path(OUT_DIR) / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
            print("retomando de", src)

In [ ]:
import gc
import json

from tanque import config
from tanque.infer import large

selected = large.select_models(MODELS, PART)
print(f"PART={PART!r}: {[large.system_of(m) for m in selected]}")

runs = []
for order, spec in enumerate(selected, start=1):
    info = large.run_spec(
        spec,
        dataset_path=config.resolve_path(f"datasets/{DATASET}.jsonl"),
        split=DATASET,
        out_dir=OUT_DIR,
        limit=LIMIT,
        max_new_tokens=MAX_NEW_TOKENS,
        retry=RETRY,
        dtype=DTYPE,
    )
    written = large.write_run_infos(
        info,
        OUT_DIR,
        repo_ref=REPO_REF,
        repo_commit=REPO_COMMIT,
        part=PART,
        session_order=order,
        gpu_names=GPU_NAMES,
    )
    print(json.dumps(info, ensure_ascii=False, indent=2))
    print("run-info:", [p.name for p in written])
    runs.append(info)
    # Libera a GPU antes do próximo modelo (run_predictions já apagou o modelo).
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# Resumo da sessão: o que rodou, o que parou (D37) e o pico de memória.
for info in runs:
    if info.get("skipped"):
        print(f"{info['system']}: PULADO ({info['skip_reason']})")
        continue
    s = info["summary"]
    r = info.get("retry_summary") or {}
    print(
        f"{info['system']}: {info['completed']}/{info['n_items']} itens"
        + (f" (parou: {info['stopped_reason']})" if info.get("stopped_reason") else "")
        + f", {info.get('quantization') or info['dtype']}, {info['gpus']} GPU(s), "
        f"pico {info['peak_gpu_mem_gib']} GiB, p50 {s['p50_latency_ms']} ms, "
        f"{r.get('retried', 0)} novas tentativas, {info['total_s'] / 60:.1f} min "
        f"[{info.get('model_source')}]"
    )

# Primeiras previsões de cada sistema, para conferir o formato e a SQL extraída.
for info in runs:
    if info.get("skipped"):
        continue
    print("==", info["system"])
    for line in Path(info["path"]).read_text(encoding="utf-8").splitlines()[:2]:
        p = json.loads(line)
        print(p["id"], p["latency_ms"], "ms |", p["error"] or p["sql"][:200].replace("\n", " "))